# 🔧 Jun tool-calling test

Isolates the one question we can't answer offline: **does Ollama parse the Jun model's
`<|tool_call>` output back into a structured `message.tool_calls` array?**

If it does, the app's tool preflight ([`chat.php`](../webapp/api/chat.php) line ~737) can work and any
remaining bug is in our PHP. If it doesn't, the model emits tool calls as raw text that Ollama
never structures, so `tool_calls` is always empty and tools silently no-op — and the fix is a
Modelfile `TEMPLATE`, not PHP.

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. First run downloads ~7 GB.

In [ ]:
#@title Step 1 — install & start Ollama
import subprocess, time, urllib.request, os, shutil

print(subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True).stdout.strip() or 'no GPU')

# Ollama's installer unpacks a .tgz with zstd — install it first or extraction fails.
subprocess.run('apt-get -qq update && apt-get -qq install -y zstd', shell=True)

# Install (do NOT silence — a silent failure is why ollama went missing before). Retry once.
for attempt in range(2):
    r = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
                       shell=True, capture_output=True, text=True)
    print(r.stdout[-1500:]); print(r.stderr[-1500:])
    if shutil.which('ollama') or os.path.exists('/usr/local/bin/ollama'):
        break
    print('retrying install...'); time.sleep(3)

OLLAMA = shutil.which('ollama') or '/usr/local/bin/ollama'
assert os.path.exists(OLLAMA), 'ollama binary still missing — read the install output above'
print('ollama at', OLLAMA)

env = os.environ.copy(); env['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen([OLLAMA, 'serve'], env=env,
                 stdout=open('/content/ollama.log','w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
        print('✅ Ollama up'); break
    except Exception:
        time.sleep(1)
else:
    print('⚠️ Ollama did not start — see /content/ollama.log')

In [ ]:
#@title Step 2 — pull the Jun model
#@markdown This is the exact model the app defaults to in chat.php.
MODEL = 'hf.co/efficiencyx/Jun-Lora-v2-GGUF:Q4_K_M'  #@param {type:'string'}
!ollama pull {MODEL}
print('\n✅ pulled', MODEL)

In [ ]:
#@title Step 3 — THE TEST: send a tools request, inspect the response
#@markdown Mirrors the app's preflight: `stream:false` + the real `search_recent_chats` schema.
import json, urllib.request

def chat(payload, timeout):
    req = urllib.request.Request('http://127.0.0.1:11434/api/chat',
                                 data=json.dumps(payload).encode(),
                                 headers={'Content-Type': 'application/json'})
    return json.load(urllib.request.urlopen(req, timeout=timeout))

# Warm-up: first inference on a 12B cold-loads into VRAM (slow on a T4). Do it once,
# with a generous timeout, so the real test below isn't racing the load.
print('warming model into VRAM (can take a couple minutes on first run)...')
chat({'model': MODEL, 'stream': False, 'messages': [{'role':'user','content':'hi'}],
      'keep_alive': '10m'}, timeout=600)
print('warm.\n')

payload = {
    'model': MODEL,
    'stream': False,
    'think': False,
    'keep_alive': '10m',
    'messages': [
        {'role': 'system', 'content': "You are Jun. Use tools when they help answer."},
        {'role': 'user', 'content': 'Can you search our past chats for what we said about pizza?'},
    ],
    'tools': [{
        'type': 'function',
        'function': {
            'name': 'search_recent_chats',
            'description': "Search saved conversation history for relevant recent messages.",
            'parameters': {
                'type': 'object',
                'properties': {
                    'query': {'type': 'string', 'description': 'What to search for in prior chats.'},
                    'limit': {'type': 'integer', 'description': 'Max messages, 1 to 8.'},
                },
                'required': ['query'],
            },
        },
    }],
}

resp = chat(payload, timeout=300)
msg = resp.get('message', {})

print('=== raw message ===')
print(json.dumps(msg, indent=2)[:2000])
print('\n=== VERDICT ===')
if msg.get('tool_calls'):
    print('✅ Ollama PARSED tool_calls — the model + Ollama support tools.')
    print('   -> Any failure is in our PHP (chat.php preflight/gating). tool_calls:')
    print('   ', json.dumps(msg['tool_calls']))
elif '<|tool_call' in msg.get('content','') or 'call:' in msg.get('content',''):
    print('❌ Model EMITTED a tool call as RAW TEXT but Ollama did NOT structure it.')
    print('   -> message.tool_calls will always be empty; tools silently no-op.')
    print('   -> Fix is a Modelfile TEMPLATE, not chat.php.')
else:
    print('⚠️ Model chose not to call a tool at all (just answered). Content:')
    print('   ', msg.get('content','')[:500])

In [ ]:
#@title Step 4 (optional) — control: does a known tool-capable model parse here?
#@markdown Confirms the test harness itself is correct. llama3.1 is small and reliably parses tool_calls.
import json, urllib.request
!ollama pull llama3.1:8b
p = dict(payload); p['model'] = 'llama3.1:8b'
req = urllib.request.Request('http://127.0.0.1:11434/api/chat',
                             data=json.dumps(p).encode(),
                             headers={'Content-Type': 'application/json'})
m = json.load(urllib.request.urlopen(req, timeout=120)).get('message', {})
print('control tool_calls:', json.dumps(m.get('tool_calls')))
print('✅ harness works' if m.get('tool_calls') else '⚠️ harness/prompt issue — revisit before trusting Step 3')

In [ ]:
#@title Step 5 — verify the new "Let me check → tool → answer" flow
#@markdown Reproduces chat.php's reworked preflight: does the model emit a lead line
#@markdown before calling the tool, and a real answer after the tool result?
import json, urllib.request

TOOL_INSTR = ("When you decide to call a tool, FIRST say one short natural line telling "
              "the user you're about to check (e.g. \"Let me check.\"), THEN call the tool. "
              "After the tool result comes back, give your real answer.")

msgs = [
    {'role': 'system', 'content': "You are Jun, a warm girlfriend AI. " + TOOL_INSTR},
    {'role': 'user', 'content': "Who is the current president of France?"},
]
tools = [{'type':'function','function':{
    'name':'web_fetch','description':'Fetch a public URL for current real-world data.',
    'parameters':{'type':'object','properties':{'url':{'type':'string'}},'required':['url']}}}]

# Phase 1 — preflight (stream:false, tools). Capture lead line + tool call.
r1 = chat({'model':MODEL,'stream':False,'think':False,'keep_alive':'10m',
           'messages':msgs,'tools':tools}, timeout=300)['message']
lead = (r1.get('content') or '').strip()
calls = r1.get('tool_calls') or []
print('LEAD LINE   :', repr(lead))
print('TOOL CALLS  :', json.dumps(calls))

# Simulate the tool result the app would compute, then Phase 2 — final answer.
if calls:
    tool_result = json.dumps({'status':200,'text':'Emmanuel Macron is the current president of France.'})
    msgs2 = msgs + [
        {'role':'assistant','content':lead,'tool_calls':calls},
        {'role':'tool','name':calls[0]['function']['name'],
         'tool_call_id':calls[0].get('id',''),'content':tool_result},
    ]
    r2 = chat({'model':MODEL,'stream':False,'think':False,'keep_alive':'10m',
               'messages':msgs2}, timeout=300)['message']
    print('FINAL ANSWER:', repr((r2.get('content') or '').strip()))
    print('\n=== What Anon would see ===')
    print((lead + '\n\n' if lead else '') + (r2.get('content') or '').strip())
else:
    print('\n⚠️ model did not call the tool for this prompt — try a more clearly live-data question')


---
## Step 6 — End-to-end: drive the REAL `chat.php`

Steps 3–5 test the model directly. This section runs the **actual edited PHP** from the
`test/tool-preamble` branch: clones the repo, starts `php -S`, seeds a logged-in user + a
prior "pizza" chat, then streams a real `/api/chat.php` request so you see the live SSE:
the **per-tool lead line**, the `tool_status` indicator, the tool running, and the answer.


In [ ]:
#@title Step 6a — clone branch, start php -S, seed a logged-in user + prior "pizza" chat
import os, subprocess, time, urllib.request, json

BRANCH = 'test/tool-preamble'  #@param {type:'string'}
REPO   = '/content/Jun'
DOCROOT = REPO + '/webapp'

# Fresh clone of the branch under test.
subprocess.run(['pkill','-9','-f','php -S'], check=False)
subprocess.run(f'rm -rf {REPO}', shell=True)
subprocess.run(['git','clone','-q','--depth','1','--branch',BRANCH,
                'https://github.com/efficiencyx/JunOS.git', REPO], check=True)

# PHP + the extensions the app needs (pdo_sqlite, curl, mbstring).
subprocess.run('apt-get -qq install -y php-cli php-sqlite3 php-curl php-mbstring',
               shell=True)

# nomic-embed-text — chat.php embeds every message; without it those calls just log & skip.
subprocess.run(['ollama','pull','nomic-embed-text'], check=False)

# Writable state/memory dirs so SQLite + the memory tool work.
STATE = '/content/state'; MEM = '/content/memory'
os.makedirs(STATE, exist_ok=True); os.makedirs(MEM, exist_ok=True)
srv_env = os.environ.copy()
srv_env.update(OLLAMA_URL='http://127.0.0.1:11434',
               OMEGA_STATE_DIR=STATE, MEMORY_DIR=MEM)

# Seed the DB via the app's own _lib.php (runs migrations, shares the same state dir).
seed = f'''<?php
require "{DOCROOT}/api/_lib.php";
$db = db(); $now = time();
$db->prepare("INSERT INTO users (email,password_hash,role,adult_consent_at,created_at) VALUES (?,?,?,?,?)")
   ->execute(["test@example.com","x","user",$now,$now]);
$uid = (int)$db->lastInsertId();
$db->prepare("INSERT INTO sessions (token,user_id,created_at,expires_at) VALUES (?,?,?,?)")
   ->execute(["testtoken",$uid,$now,$now+86400]);
$db->prepare("INSERT INTO conversations (user_id,title,created_at,updated_at) VALUES (?,?,?,?)")
   ->execute([$uid,"old chat",$now-9000,$now-9000]);
$c1=(int)$db->lastInsertId();
$db->prepare("INSERT INTO messages (conversation_id,role,content,created_at) VALUES (?,?,?,?)")
   ->execute([$c1,"user","I love pepperoni pizza from Tony's on 5th street",$now-9000]);
$db->prepare("INSERT INTO conversations (user_id,title,created_at,updated_at) VALUES (?,?,?,?)")
   ->execute([$uid,null,$now,$now]);
echo (int)$db->lastInsertId();
'''
open('/content/seed.php','w').write(seed)
CONV_ID = int(subprocess.run(['php','/content/seed.php'], env=srv_env,
                             capture_output=True, text=True).stdout.strip())
print('seeded, current conversation_id =', CONV_ID)

# Start the app.
subprocess.Popen(['php','-S','127.0.0.1:8000','-t',DOCROOT], env=srv_env,
                 stdout=open('/content/php.log','w'), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/api/models.php', timeout=2); break
    except Exception: time.sleep(0.5)
print('✅ php server up on :8000')


In [ ]:
#@title Step 6b — stream a real /api/chat.php request and watch the live events
#@markdown Sends a recall-style message. Expected sequence: a `debug` event, then a
#@markdown **lead-line token** (model's or the per-tool fallback), then `tool_status`
#@markdown running→done while `search_recent_chats` runs, then the answer tokens.
import json, urllib.request

USER_MSG = "hey, what did we say about pizza before?"  #@param {type:'string'}

body = json.dumps({
    'messages': [{'role': 'user', 'content': USER_MSG}],
    'model': MODEL,
    'conversation_id': CONV_ID,
    'reasoning': 'low',
    'client_time': 'Monday, July 14, 2026 at 3:00 PM',
}).encode()

req = urllib.request.Request('http://127.0.0.1:8000/api/chat.php', data=body,
                             headers={'Content-Type': 'application/json',
                                      'Cookie': 'omega_session=testtoken'})

reply, lead_seen, tools_seen = '', '', []
print('--- live SSE from chat.php ---')
with urllib.request.urlopen(req, timeout=600) as r:
    for raw in r:
        line = raw.decode('utf-8', 'replace').rstrip('\n')
        if not line.startswith('data:'):
            continue
        payload = line[5:].strip()
        if payload == '[DONE]':
            print('[DONE]'); break
        try:
            obj = json.loads(payload)
        except Exception:
            continue
        if 'debug' in obj:
            print('· debug (system prompt + live context assembled)')
        elif 'tool_status' in obj:
            ts = obj['tool_status']; tools_seen.append(ts)
            print(f"· 🔧 tool {ts.get('name')} -> {ts.get('state')}")
        elif 'thinking' in obj:
            pass
        elif 'token' in obj:
            tok = obj['token']
            reply += tok
            if not reply.strip().replace(tok.strip(), ''):  # first visible chunk = lead line
                lead_seen = lead_seen or tok.strip()
            print('· token:', repr(tok))
        elif 'stats' in obj:
            print('· stats:', obj['stats'].get('eval_count'), 'tokens')
        elif 'error' in obj:
            print('· ⚠️ error:', obj['error'])

print('\n=== RESULT ===')
print('lead line seen   :', repr(lead_seen))
print('tool events seen :', tools_seen)
print('full reply:\n', reply.strip())
print('\n(peek at server log if anything looks off:)')
!tail -n 15 /content/php.log
